# NyayaAI - Training Notebook

End-to-end pipeline for **IndiaLex-FT** — a LoRA fine-tuned LLM for Indian legal and income-tax Q&A.

**Steps covered:**
1. Install dependencies
2. Load API keys from `.env`
3. Baseline evaluation (pre-training)
4. LoRA SFT fine-tuning
5. Post-training evaluation
6. LLM-as-Judge scoring via claude-sonnet-4-6
7. Download outputs

> **Note:** Run this notebook from the `indialex-ft/` directory. Ensure your `.env` file has `ANTHROPIC_API_KEY` and `GROQ_API_KEY` set before starting.

In [ ]:
import os
from pathlib import Path

# Ensure the notebook working directory is indialex-ft/
notebook_dir = Path("__file__" if "__file__" in dir() else ".").resolve()
print(f"Working directory: {os.getcwd()}")

In [2]:
!pip install -r requirements.txt --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.7/960.7 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 109.9 MB/s eta 0:00:00


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads indialex-ft/.env

def _check_secret(name, required=True):
    val = os.environ.get(name)
    if val:
        print(f"{name} loaded: True")
    else:
        status = "MISSING (required — add to .env)" if required else "skipped (optional)"
        print(f"{name} loaded: {status}")

_check_secret("ANTHROPIC_API_KEY", required=True)
_check_secret("GROQ_API_KEY",      required=True)
_check_secret("HF_TOKEN",          required=True)   # needed: Llama 3.2 is a gated model
_check_secret("WANDB_API_KEY",     required=False)

In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("HuggingFace login successful.")
else:
    raise EnvironmentError(
        "HF_TOKEN is not set. Llama 3.2 is a gated model — "
        "accept the license at huggingface.co/meta-llama/Llama-3.2-3B-Instruct "
        "and add HF_TOKEN=hf_... to your .env file."
    )

## Step 1: Baseline Evaluation

Runs inference on the test split using the **base model** (before fine-tuning).
Computes ROUGE-L and BERTScore F1, and saves results to `evals/baseline_results.json`.

In [5]:
!python scripts/baseline_eval.py

INFO  Loading tokenizer and model: meta-llama/Llama-3.2-3B-Instruct
INFO  HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO  HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json "HTTP/1.1 401 Unauthorized"
INFO  HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json "HTTP/1.1 401 Unauthorized"
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 772, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '401 Unauthorized' for url 'https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

## Step 2: Training

Fine-tunes the base model with **LoRA + QLoRA (4-bit NF4)** via TRL SFTTrainer.
Saves the LoRA adapter to `outputs/adapter/` and the merged model to `outputs/merged/`.
Logs metrics to Weights & Biases.

In [ ]:
!python scripts/train.py

## Step 3: Post-training Evaluation

Runs inference on the test split using the **fine-tuned merged model**.
Computes ROUGE-L and BERTScore F1, compares against baseline, and saves
a delta summary to `evals/ft_results.json`.

In [ ]:
!python scripts/evaluate.py

## Step 4: LLM-as-Judge Evaluation

Uses **claude-sonnet-4-6** (Anthropic API) to score each fine-tuned answer on
four legal-domain dimensions (1–5 each): correctness, faithfulness, helpfulness,
and legal accuracy. Results saved to `evals/judge_results.json`.

In [ ]:
!python evals/llm_judge.py --results evals/ft_results.json

In [ ]:
import shutil
from pathlib import Path

for name, base_dir in [("nyayaai_outputs", "outputs"), ("nyayaai_evals", "evals")]:
    if Path(base_dir).exists():
        shutil.make_archive(name, "zip", root_dir=".", base_dir=base_dir)
        print(f"Zipped {base_dir}/ → {name}.zip")
    else:
        print(f"Skipped {base_dir}/ — directory does not exist yet.")

try:
    from google.colab import files
    for f in ["nyayaai_outputs.zip", "nyayaai_evals.zip"]:
        if Path(f).exists():
            files.download(f)
except ImportError:
    print("Zip files saved locally (not running in Colab).")